In [0]:
dbutils.widgets.text("Environment", "dev", "Select the current environment/catalog")
dbutils.widgets.text("Host", "", "Databricks WS Host")
dbutils.widgets.text("AccessToken, "", "Secure Access Token")

In [0]:
env = dbutils.widgets.get("Environment")
host = dbutils.widgets.get("Host")
token = dbutils.widgets.get("AccessToken")

In [0]:
%run ./02-setup

In [0]:
SH = SetupHelper
SH.cleanup()

In [0]:
job_payload = \
{
    "name": "stream-test",
    "webhook_notifications": {},
    "timeout_seconds": 0,
    "max_concurrent_runs": 1,
    "tasks": [
        {
            "task_key": "stream-test-task",
            "run_if": "ALL_SUCCESS",
            "notebook_task": {
                "notebook_path": f"/Repos/databricks-capstone/databricks-sbit/07-run",
                "source": "WORKSPACE"
            },
            "job_cluster_key": "stream-test-cluster",
            "timeout_seconds": 0,
            "email_notifications": {}
        }
    ],
    "job_clusters": [
        {
            "job_cluster_key": "stream-test-cluster",
            "new_cluster": {
                "spark_version": "12.2.x-cpu-ml-scala2.12",
                "spark_conf": {
                    "spark.databricks.delta.preview.enabled": "true",
                    "spark.master": "local[*, 4]",
                    "spark.databricks.cluster.profile": "singleNode"
                },
                "azure_attributes": {
                    "first_on_demand": 1,
                    "availability": "ON_DEMAND_AZURE",
                    "spot_bid_max_price": 0
                },
                "node_type_id": "Standard_DS3_v2",
                "driver_node_type_id": "Standard_DS3_v2",
                "custom_tags": {
                    "ResourceClass": "SingleNode"
                },
                "data_security_mode": "SINGLE_USER",
                "runtime_engine": "STANDARD",
                "num_workers": 0
            }
        }
    ],
    "format": "MULTI_TASK"
}


In [0]:
# create a streaming job
import requests
import json

create_response = requests.post(host + "/api/2.1/jobs/create", data=json.dumps(job_payload), headers={"Authorization": f"Bearer {token}"})
print(f"Response: {create_response.json()}")
job_id = json.loads(create_response.content.decode("utf-8"))["job_id"]



In [0]:
# trigger the streaming job
run_payload = {
    "job_id": job_id,
    "notebook_params": {
        "env": env,
        "RunType": "stream",
        "ProcessingTime": "1 seconds"
    }    
}
run_response = requests.post(host + "/api/2.1/jobs/run-now", data=json.dumps(run_payload), headers={"Authorization": f"Bearer {token}"})
print(f"Response: {run_response.json()}")
run_id = json.loads(run_response.content.decode("utf-8"))["run_id"]
print(f"Started Job {job_id} with Run {run_id}")

In [0]:
# wait for job to start
import time
status_payload = {
    "run_id": run_id
}
job_status = "PENDING"
while job_status != "RUNNING":
    status_response = requests.get(host + "/api/2.1/jobs/runs/get", params=status_payload, headers={"Authorization": f"Bearer {token}"})
    job_status = json.loads(status_response.content.decode("utf-8"))["state"]["life_cycle_state"]
    print(f"Job run {job_id}/{run_id} status is {job_status}")
    time.sleep(5)

In [0]:
%run ./03-history-loader

In [0]:
%run ./08-test-data-producer

In [0]:
%run ./04-bronze-ingestions

In [0]:
%run ./05-silver-transformations

In [0]:
%run ./06-gold-layer

In [0]:
import time

print("Sleep for 2 minutes and ket setup and history loader to finish...")
time.sleep(120)

HL = HistoryLoader(env)
PR = Producer()
BZ = Bronze(env)
SV = Silver(env)
GD = Gold(env)

SH.validate()
HL.validate()

# produce some incremental data stream
PR.produce(1)
PR.validate(1)

In [0]:
print("Sleep for 2 minutes to allow data ingestion to finish")
time.sleep(120)

# Validate DB content
BZ.validate(1)
SV.validate(1)
GD.validate(1)

In [0]:
# Produce new incremental data and wait for micro batch
PR.produce(2)
PR.validate(2)
print("Sleep for 2 minutes to allow data ingestion to finish")

# Validate DB content after the second set
BZ.validate(2)
SV.validate(2)
GD.validate(2)

In [0]:
# terminate the streaming job
cancel_payload = {
    "run_id": run_id
}
cancel_response = requests.post(host + "/api/2.1/jobs/runs/cancel", data=json.dumps(cancel_payload), headers={"Authorization": f"Bearer {token}"})
print(f"Response: {cancel_response.json()}")

In [0]:
# delete the job
delete_payload = {
    "job_id": job_id
}
delete_response = requests.delete(host + "/api/2.1/jobs/delete", data=json.dumps(delete_payload), headers={"Authorization": f"Bearer {token}"})
print(f"Response: {delete_response.json()}")

In [0]:
dbutils.notebook.exit("SUCCESS")